<a href="https://colab.research.google.com/github/kilianodonell-cmd/Durban/blob/main/Field_Map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Install packages ─────────────────────────────────────────
!pip install folium rasterio geopandas numpy Pillow -q


In [ ]:

# ── Mount Google Drive ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CELL 1 — Setup (IMPROVED with file checks)
# Field Map — Housing Suitability
# Mzinyati Stream Catchment, eThekwini Municipality
# ============================================================

import os, json, numpy as np
import geopandas as gpd
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
import folium
from folium import LayerControl
from PIL import Image
import base64, io

CONFIG_PATH = '/content/drive/MyDrive/Durban/outputs/field_map_config.json'

with open(CONFIG_PATH) as f:
    config = json.load(f)

OUTPUT_ROOT    = config['OUTPUT_ROOT']
TARGET_CRS     = config['TARGET_CRS']
SCENARIOS      = config['SCENARIOS']
SUIT_COLORS    = config['SUIT_COLORS']
SUIT_LABELS    = config['SUIT_LABELS']
RASTERS_DIR    = os.path.join(OUTPUT_ROOT, 'rasters')
BUILDINGS_PATH = os.path.join(OUTPUT_ROOT, 'outputs', 'at_risk_buildings.gpkg')

print("=" * 60)
print("FIELD MAP SETUP")
print("=" * 60)
print(f"  Output root: {OUTPUT_ROOT}")
print(f"  Scenarios:   {list(SCENARIOS.keys())}")

# Check that required files exist
print("\n  Checking files...")
missing = []

for scenario in SCENARIOS:
    raster_path = os.path.join(RASTERS_DIR, f"suitability_{scenario}.tif")
    if os.path.exists(raster_path):
        print(f"    ✓ suitability_{scenario}.tif")
    else:
        print(f"    ✗ suitability_{scenario}.tif MISSING")
        missing.append(raster_path)

constraint_path = os.path.join(RASTERS_DIR, 'constraint_mask.tif')
if os.path.exists(constraint_path):
    print(f"    ✓ constraint_mask.tif")
else:
    print(f"    ✗ constraint_mask.tif MISSING")
    missing.append(constraint_path)

if os.path.exists(BUILDINGS_PATH):
    print(f"    ✓ at_risk_buildings.gpkg")
else:
    print(f"    ✗ at_risk_buildings.gpkg MISSING")
    missing.append(BUILDINGS_PATH)

if missing:
    print(f"\n  ⚠️ WARNING: {len(missing)} files missing. Run MCA notebook first.")
else:
    print(f"\n  ✓ All files ready. Proceed to CELL 2.")

print("=" * 60)

In [ ]:
# ============================================================
# CELL 2A-1: Setup & Load Data
# ============================================================

print("=" * 60)
print("CELL 2A-1: SETUP & LOAD DATA")
print("=" * 60)

# Settings
WGS_RASTERS_DIR = os.path.join(RASTERS_DIR, "wgs84")
PREVIEW_SCENARIO = list(SCENARIOS.keys())[0]  # hazard_focused
SHOW_BUILDINGS_PREVIEW = True

print(f"  Scenario: {PREVIEW_SCENARIO} = {SCENARIOS[PREVIEW_SCENARIO]}")
print(f"  Show buildings: {SHOW_BUILDINGS_PREVIEW}")

# Paths
preview_raster_path = os.path.join(RASTERS_DIR, f"suitability_{PREVIEW_SCENARIO}.tif")
constraint_wgs_path = os.path.join(WGS_RASTERS_DIR, "constraint_mask_wgs84.tif")

# Check files exist
print("\n  Checking files...")
print(f"    Suitability: {os.path.exists(preview_raster_path)}")
print(f"    Constraint: {os.path.exists(constraint_wgs_path)}")
print(f"    Buildings: {os.path.exists(BUILDINGS_PATH)}")

# Load AOI
catchments = gpd.read_file(config['CATCHMENTS_PATH'])
aoi = catchments[catchments[config['AOI_FIELD']] == config['AOI_VALUE']]
aoi_wgs84 = aoi.to_crs("EPSG:4326")

bounds = aoi_wgs84.total_bounds
centre = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]
folium_bounds = [[bounds[1], bounds[0]], [bounds[3], bounds[2]]]

print(f"\n  AOI centre: {centre[0]:.4f}, {centre[1]:.4f}")
print("  ✓ Setup complete")

In [ ]:
# ============================================================
# CELL 2A-2: Load Suitability Raster
# ============================================================

print("=" * 60)
print("CELL 2A-2: LOAD SUITABILITY RASTER")
print("=" * 60)

with rasterio.open(preview_raster_path) as src:
    preview_data = src.read(1).astype("float32")
    preview_transform = src.transform
    preview_crs = src.crs

preview_data[preview_data <= 0] = np.nan

unique_vals = np.unique(preview_data[~np.isnan(preview_data)])
print(f"  Raster shape: {preview_data.shape}")
print(f"  Unique values: {unique_vals}")
print(f"  Correctly classified (1-5)? {set(unique_vals).issubset({1,2,3,4,5})}")
print("  ✓ Suitability raster loaded")

In [ ]:
# ============================================================
# CELL 2A-3: Load & Resample Constraint Raster
# ============================================================

print("=" * 60)
print("CELL 2A-3: LOAD & RESAMPLE CONSTRAINT RASTER")
print("=" * 60)

with rasterio.open(constraint_wgs_path) as src:
    constraint_data_original = src.read(1).astype("float32")
    constraint_transform = src.transform
    constraint_crs = src.crs

print(f"  Original constraint shape: {constraint_data_original.shape}")

# Resample to match suitability raster
constraint_resampled = np.zeros(preview_data.shape, dtype='float32')

reproject(
    source=constraint_data_original,
    destination=constraint_resampled,
    src_transform=constraint_transform,
    src_crs=constraint_crs,
    dst_transform=preview_transform,
    dst_crs=preview_crs,
    resampling=Resampling.nearest
)

constraint_mask = (constraint_resampled == 1) | (constraint_resampled > 0)
n_constraint_cells = constraint_mask.sum()

print(f"  Resampled constraint shape: {constraint_resampled.shape}")
print(f"  Constraint cells: {n_constraint_cells:,}")
print("  ✓ Constraint raster loaded and resampled")

In [ ]:
# ============================================================
# CHECK CONSTRAINT MASK VALUES
# ============================================================

print("=" * 60)
print("CHECKING CONSTRAINT MASK")
print("=" * 60)

# Check what values are in constraint mask
print(f"\nUnique values in constraint_resampled: {np.unique(constraint_resampled)}")

# Check distribution
unique, counts = np.unique(constraint_resampled, return_counts=True)
for val, count in zip(unique, counts):
    print(f"  Value {val}: {count:,} cells")

# Check what's being masked
print(f"\nconstraint_mask (True = constrained):")
print(f"  True count: {constraint_mask.sum():,}")
print(f"  False count: {(~constraint_mask).sum():,}")

# Check a sample of preview_overridden values in constraint areas
constrained_cells = preview_overridden[constraint_mask]
print(f"\nValues in constraint areas after override:")
unique_constrained = np.unique(constrained_cells)
print(f"  Unique values: {unique_constrained}")

if 5 in unique_constrained:
    print("  ✓ Class 5 IS present in constraint areas")
else:
    print("  ✗ Class 5 is NOT in constraint areas - OVERRIDE FAILED")

# Check cells that are NOT constraints
unconstrained_cells = preview_overridden[~constraint_mask]
print(f"\nValues in non-constraint areas:")
print(f"  Unique values: {np.unique(unconstrained_cells)}")


In [ ]:
# ============================================================
# CELL 2A-4a: Create AOI Mask
# ============================================================

print("=" * 60)
print("CELL 2A-4a: CREATE AOI MASK")
print("=" * 60)

from rasterio import features

# Get AOI geometry in same CRS as raster
aoi_projected = aoi.to_crs(preview_crs)

# Create AOI mask raster
aoi_mask_raster = features.rasterize(
    [(geom, 1) for geom in aoi_projected.geometry],
    out_shape=preview_data.shape,
    transform=preview_transform,
    fill=0,
    dtype='uint8'
)

aoi_mask = (aoi_mask_raster == 1)

print(f"  Raster shape: {preview_data.shape}")
print(f"  AOI cells (should be 50,460): {aoi_mask.sum():,}")
print(f"  Outside AOI cells: {(~aoi_mask).sum():,}")

# Visual check - what percentage of raster is AOI?
total_cells = preview_data.shape[0] * preview_data.shape[1]
print(f"  AOI % of total raster: {aoi_mask.sum()/total_cells*100:.1f}%")

In [ ]:
# ============================================================
# EXPLAIN CONSTRAINT CELLS - Using CELL 11 output
# ============================================================

print("=" * 60)
print("WHY 122,957 CELLS ARE CONSTRAINED")
print("=" * 60)

# Load the constraint mask from CELL 11
constraint_path = os.path.join(RASTERS_DIR, 'constraint_mask.tif')

with rasterio.open(constraint_path) as src:
    constraint_mask_original = src.read(1).astype('float32')

print(f"\nConstraint mask shape: {constraint_mask_original.shape}")
print(f"Unique values: {np.unique(constraint_mask_original)}")
print(f"  Value 0 = buildable: {(constraint_mask_original == 0).sum():,} cells")
print(f"  Value 1 = constrained: {(constraint_mask_original == 1).sum():,} cells")
print(f"  Value 255 = NoData: {(constraint_mask_original == 255).sum():,} cells")

# Now resample to match preview raster (like in CELL 2A-3)
print("\n" + "=" * 60)
print("RESAMPLED TO SUITABILITY RASTER")
print("=" * 60)

# Resample to match preview_data shape
constraint_resampled_check = np.zeros(preview_data.shape, dtype='float32')

reproject(
    source=constraint_mask_original,
    destination=constraint_resampled_check,
    src_transform=src.transform,
    src_crs=src.crs,
    dst_transform=preview_transform,
    dst_crs=preview_crs,
    resampling=Resampling.nearest
)

print(f"\nAfter resampling to {preview_data.shape}:")
print(f"  Value 0 (buildable): {(constraint_resampled_check == 0).sum():,} cells")
print(f"  Value 1 (constrained): {(constraint_resampled_check == 1).sum():,} cells")
print(f"  Value 255 (NoData): {(constraint_resampled_check == 255).sum():,} cells")

# The 122,957 comes from value 1 cells
print(f"\n✓ CONSTRAINED CELLS = {(constraint_resampled_check == 1).sum():,}")
print(f"  This matches your CELL 11 output (24,135 buildable, 26,325 constrained in AOI)")